
# Assignment 04 · Miền MNIST · Notebook 01: Mạng tích chập hai chiều viết hoàn toàn bằng NumPy

**Học phần:** Phát triển các Hệ thống Thông minh, Học viện Công nghệ Bưu chính Viễn thông

**Sinh viên:** Nguyễn Duy Nghĩa &nbsp;·&nbsp; **Mã sinh viên:** B23DCCN600 &nbsp;·&nbsp; **Lớp:** D23CTPM01

**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế

**Học kỳ:** Học kỳ 1 năm học 2026 &ndash; 2027

---

## Mục tiêu notebook

Đây là notebook trọng tâm của miền MNIST. Toàn bộ mạng tích chập hai chiều được xây dựng từ
con số không, chỉ dùng NumPy, không sử dụng bất kỳ thư viện học sâu nào. Cụ thể báo cáo sẽ:

1. Suy dẫn đầy đủ cơ sở toán học của từng tầng: tích chập hai chiều, công thức kích thước đầu
   ra, ReLU, max pooling, tầng kết nối đầy đủ, softmax kèm thủ thuật trừ cực đại, hàm mất mát
   entropy chéo và thuật toán Adam.
2. Hiện thực phép biến đổi `im2col` và `col2im` bằng chỉ số véc-tơ hóa hoàn toàn, nhờ đó phép
   tích chập quy về một phép nhân ma trận duy nhất (GEMM). Không có bất kỳ vòng lặp Python nào
   chạy trên từng điểm ảnh.
3. Kiểm chứng phép tích chập bằng một **ví dụ số làm tay** rồi đối chiếu với kết quả của lớp
   `Conv2D` đã cài.
4. Thực hiện **kiểm tra gradient bằng sai phân hữu hạn** trên toàn bộ tham số học được của
   mạng, in ra bảng so sánh gradient giải tích với gradient số và sai số tương đối. Đây là kết
   quả then chốt chứng minh phép lan truyền ngược viết tay là đúng.
5. Huấn luyện hai biến thể bắt buộc theo mục 4 hợp đồng: *Baseline* và *Improved*, rồi định
   lượng mức cải thiện theo điểm phần trăm tuyệt đối.

Ba hình bắt buộc notebook này sinh ra: `fig_mnist_scratch_curves.png`,
`fig_mnist_scratch_confusion.png`, `fig_mnist_scratch_comparison.png`.

> **Ghi chú về ngân sách tính toán.** Mạng NumPy thuần chạy trên CPU chậm hơn một bậc độ lớn so
> với PyTorch vì không có nhân GEMM đa luồng tối ưu và không tận dụng được vector hóa mức
> thanh ghi. Huấn luyện trên toàn bộ 48 000 ảnh sẽ vượt xa ngân sách 15 phút mỗi notebook mà
> hợp đồng quy định. Vì vậy hai mô hình NumPy được huấn luyện trên **tập con phân tầng
> 12 000 ảnh train và 3 000 ảnh validation**, nhưng **được đánh giá trên trọn vẹn 10 000 ảnh
> của tập kiểm thử gốc**, đúng bằng tập mà PyTorch và Keras sẽ dùng ở notebook 02. Nhờ đó
> phép so sánh ba cách cài đặt vẫn công bằng ở phía đánh giá. Sai lệch này được ghi rõ trong
> khóa `notes` của tệp metrics và trong chú thích của mọi hình liên quan.


## 1. Nhập thư viện và cấu hình seed

In [1]:

import os, json, time, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/mnist.npz'
FIG_DIR   = '../reports/figures'
REP_DIR   = '../reports'
os.makedirs(FIG_DIR, exist_ok=True)

# Kích thước tập con dùng cho mô hình NumPy thuần (xem ghi chú ngân sách tính toán)
N_SUB_TRAIN = 12000
N_SUB_VAL   = 3000
EPOCHS      = 12
BATCH_SIZE  = 64

print('NumPy', np.__version__)
print(f'Tập con NumPy: train={N_SUB_TRAIN}, val={N_SUB_VAL}, epochs={EPOCHS}, batch={BATCH_SIZE}')

NumPy 2.4.0
Tập con NumPy: train=12000, val=3000, epochs=12, batch=64



## 2. Nạp dữ liệu, chia tập và chuẩn hóa

Quy trình tiền xử lý tuân thủ mục 3 của hợp đồng tích hợp và **lý do** của từng bước như sau.

**Bước chia train/validation.** Tập 60 000 ảnh gốc được tách theo tỉ lệ 80/20 với
`stratify=y` và `random_state=42`. Phân tầng đảm bảo mười lớp giữ nguyên tỉ lệ ở cả hai nhánh,
nếu không, một lớp thưa như chữ số 5 có thể bị lấy mẫu lệch và làm nhiễu tín hiệu chọn epoch.
Tập kiểm thử 10 000 ảnh **không bao giờ** được dùng để chọn epoch hay điều chỉnh siêu tham số;
nó chỉ được chạm tới đúng một lần ở bước đánh giá cuối.

**Bước đưa về thang $[0,1]$.** Phép chia cho 255 biến số nguyên `uint8` thành số thực. Nếu
giữ nguyên thang $[0,255]$, tích vô hướng ở tầng đầu sẽ có biên độ lớn gấp 255 lần, đẩy
logit ra vùng bão hòa của softmax ngay ở bước lặp đầu tiên.

**Bước chuẩn hóa trung bình và độ lệch chuẩn.** Ta trừ $\mu$ và chia $\sigma$ học từ nhánh
train. Việc này đưa dữ liệu về trung bình 0, qua đó tránh hiện tượng toàn bộ gradient của một
bộ lọc cùng dấu, vốn buộc quỹ đạo tối ưu đi hình chữ chi. Hai hằng số bắt buộc học từ train để
không rò rỉ thông tin của tập kiểm thử.

**Bước lấy tập con phân tầng cho mô hình NumPy.** Dùng tiếp `train_test_split` với `stratify`
để rút 12 000 ảnh từ nhánh train và 3 000 ảnh từ nhánh validation, giữ nguyên tỉ lệ lớp.

In [2]:

_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)

# --- Chia train / validation có phân tầng ---
idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)

x_tr_u8, y_tr_full = x_train_raw[idx_tr], y_train_raw[idx_tr]
x_va_u8, y_va_full = x_train_raw[idx_va], y_train_raw[idx_va]

# --- Hằng số chuẩn hóa học từ nhánh train ---
MEAN = float((x_tr_u8.astype(np.float32) / 255.0).mean())
STD  = float((x_tr_u8.astype(np.float32) / 255.0).std())
print(f'MEAN = {MEAN:.10f}   STD = {STD:.10f}   (học từ {len(idx_tr)} ảnh nhánh train)')

def preprocess(x_u8):
    '''uint8 (N,28,28) -> float32 (N,1,28,28) đã chuẩn hóa theo MEAN/STD của train.'''
    x = x_u8.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return x[:, None, :, :]

X_tr_full = preprocess(x_tr_u8)
X_va_full = preprocess(x_va_u8)
X_te      = preprocess(x_test_raw)
y_te      = y_test_raw

print(f'X_tr_full {X_tr_full.shape} | X_va_full {X_va_full.shape} | X_te {X_te.shape}')

# --- Tập con phân tầng cho mô hình NumPy thuần ---
sub_tr, _ = train_test_split(np.arange(len(y_tr_full)), train_size=N_SUB_TRAIN,
                             stratify=y_tr_full, random_state=RANDOM_SEED)
sub_va, _ = train_test_split(np.arange(len(y_va_full)), train_size=N_SUB_VAL,
                             stratify=y_va_full, random_state=RANDOM_SEED)

X_tr, y_tr = X_tr_full[sub_tr], y_tr_full[sub_tr]
X_va, y_va = X_va_full[sub_va], y_va_full[sub_va]

print()
print(f'Tập con train NumPy : {X_tr.shape}  phân phối lớp {np.bincount(y_tr, minlength=10).tolist()}')
print(f'Tập con val   NumPy : {X_va.shape}  phân phối lớp {np.bincount(y_va, minlength=10).tolist()}')
print(f'Tập kiểm thử (đầy đủ): {X_te.shape} phân phối lớp {np.bincount(y_te, minlength=10).tolist()}')
print()
print(f'Khoảng giá trị sau chuẩn hóa: [{X_tr.min():.4f}, {X_tr.max():.4f}]')
print(f'Trung bình / độ lệch chuẩn tập con train sau chuẩn hóa: '
      f'{X_tr.mean():.6f} / {X_tr.std():.6f}')

MEAN = 0.1307886839   STD = 0.3082403541   (học từ 48000 ảnh nhánh train)


X_tr_full (48000, 1, 28, 28) | X_va_full (12000, 1, 28, 28) | X_te (10000, 1, 28, 28)

Tập con train NumPy : (12000, 1, 28, 28)  phân phối lớp [1184, 1349, 1192, 1226, 1168, 1084, 1184, 1253, 1170, 1190]
Tập con val   NumPy : (3000, 1, 28, 28)  phân phối lớp [296, 337, 298, 307, 292, 271, 296, 313, 293, 297]
Tập kiểm thử (đầy đủ): (10000, 1, 28, 28) phân phối lớp [980, 1135, 1032, 1010, 982, 892, 958, 1028, 974, 1009]

Khoảng giá trị sau chuẩn hóa: [-0.4243, 2.8199]
Trung bình / độ lệch chuẩn tập con train sau chuẩn hóa: 0.000372 / 1.000475



**Diễn giải.** Tập con 12 000 ảnh giữ được phân phối lớp gần như y hệt tập train đầy đủ vì
phép lấy mẫu có phân tầng, lớp đông nhất và lớp thưa nhất chênh nhau cùng một tỉ lệ đã quan sát
ở notebook 00. Sau chuẩn hóa, trung bình của tập con xấp xỉ 0 và độ lệch chuẩn xấp xỉ 1, đúng
như thiết kế. Giá trị nhỏ nhất $-0{,}4243$ ứng với điểm nền đen và giá trị lớn nhất $2{,}8199$
ứng với điểm mực bão hòa, khoảng động này hoàn toàn phù hợp với giả định của khởi tạo He Normal.


## 3. Cơ sở toán học của phép tích chập hai chiều

### 3.1 Định nghĩa phép tích chập rời rạc hai chiều

Gọi $X \in \mathbb{R}^{N \times C_{in} \times H \times W}$ là khối đầu vào theo quy ước NCHW,
$W \in \mathbb{R}^{C_{out} \times C_{in} \times K \times K}$ là khối bộ lọc và
$b \in \mathbb{R}^{C_{out}}$ là véc-tơ chệch. Với bước nhảy $s$ và mức đệm $p$, đầu ra
$Z \in \mathbb{R}^{N \times C_{out} \times H_{out} \times W_{out}}$ được định nghĩa bởi

$$
Z_{n,\,o,\,i,\,j}
\;=\; b_{o} \;+\; \sum_{c=0}^{C_{in}-1} \sum_{u=0}^{K-1} \sum_{v=0}^{K-1}
W_{o,\,c,\,u,\,v}\; \tilde{X}_{n,\,c,\;i\cdot s + u,\;j\cdot s + v}
$$

trong đó $\tilde{X}$ là $X$ sau khi đệm $p$ hàng và $p$ cột số 0 ở cả bốn phía.

Cần nhấn mạnh một điểm thuật ngữ: công thức trên thực chất là phép **tương quan chéo**
(cross-correlation) chứ không phải tích chập theo nghĩa giải tích, vì bộ lọc không bị lật
$180^\circ$. Mọi thư viện học sâu hiện đại, bao gồm PyTorch và Keras, đều dùng quy ước này vì
bộ lọc là tham số học được: mạng tự học ra bộ lọc đã lật nếu cần, nên phép lật trở nên thừa.
Báo cáo giữ nguyên quy ước đó để kết quả NumPy khớp được với hai framework.

### 3.2 Công thức kích thước đầu ra

Trên mỗi chiều không gian, cửa sổ $K$ trượt trên tín hiệu đã đệm dài $H + 2p$ với bước $s$:

$$
H_{out} = \left\lfloor \frac{H + 2p - K}{s} \right\rfloor + 1,
\qquad
W_{out} = \left\lfloor \frac{W + 2p - K}{s} \right\rfloor + 1 .
$$

Hai trường hợp riêng được dùng trong báo cáo, với $K = 3$ và $s = 1$:

| Cấu hình | $p$ | $H = 28$ | Ý nghĩa |
|---|---|---|---|
| *Valid* (Baseline) | $0$ | $H_{out} = 28 - 3 + 1 = 26$ | mỗi tầng tích chập co ảnh đi 2 điểm ảnh |
| *Same* (Improved) | $1$ | $H_{out} = 28 + 2 - 3 + 1 = 28$ | kích thước không đổi, biên ảnh được giữ |

Điều kiện tổng quát để kích thước không đổi khi $s = 1$ là $p = (K-1)/2$, đúng với $K$ lẻ.
Đây là lý do các kiến trúc hiện đại gần như luôn chọn $K$ lẻ.

### 3.3 Số tham số và độ phức tạp

Một tầng tích chập có $C_{out}(C_{in}K^2 + 1)$ tham số, hoàn toàn độc lập với $H$ và $W$. Đây
chính là cơ chế **chia sẻ trọng số**: cùng một bộ lọc quét toàn ảnh nên số tham số nhỏ hơn
nhiều bậc so với tầng kết nối đầy đủ tương đương. Chi phí tính toán là
$O(N \cdot C_{out} \cdot C_{in} \cdot K^2 \cdot H_{out} \cdot W_{out})$ phép nhân cộng.


## 4. Phép biến đổi im2col và col2im

### 4.1 Ý tưởng

Viết trực tiếp sáu vòng lặp lồng nhau theo công thức ở mục 3.1 sẽ cho một cài đặt đúng nhưng
chậm không thể chấp nhận trong Python. Giải pháp kinh điển là **im2col**: trải mọi khối cục bộ
$C_{in} \times K \times K$ mà cửa sổ tích chập nhìn thấy thành một cột, xếp cạnh nhau thành ma
trận

$$
X_{col} \in \mathbb{R}^{(C_{in}K^2) \times (N \cdot H_{out} W_{out})} .
$$

Đồng thời trải khối bộ lọc thành $W_{row} \in \mathbb{R}^{C_{out} \times (C_{in}K^2)}$. Khi đó
toàn bộ phép tích chập rút gọn thành **một phép nhân ma trận duy nhất**:

$$
Z_{col} \;=\; W_{row} \, X_{col} \;+\; b\,\mathbf{1}^{\top},
\qquad Z_{col} \in \mathbb{R}^{C_{out} \times (N \cdot H_{out} W_{out})} ,
$$

rồi định dạng lại $Z_{col}$ về $(N, C_{out}, H_{out}, W_{out})$. Cái giá phải trả là bộ nhớ:
mỗi điểm ảnh bị nhân bản tối đa $K^2$ lần trong $X_{col}$. Đổi lại ta được tốc độ của thư viện
BLAS đã tối ưu suốt nhiều thập kỷ.

### 4.2 Xây dựng chỉ số một lần, dùng lại nhiều lần

Thay vì cắt lát trong vòng lặp, ta dựng sẵn ba mảng chỉ số $(k, i, j)$ mô tả chính xác vị trí
kênh, hàng, cột của từng phần tử trong $X_{col}$, sau đó lấy dữ liệu bằng một lần lập chỉ mục
nâng cao (`fancy indexing`). Với hàng thứ $r$ của $X_{col}$, ta phân rã

$$
r = c\,K^2 + u\,K + v
\;\Longrightarrow\;
c = \left\lfloor \frac{r}{K^2} \right\rfloor,\quad
u = \left\lfloor \frac{r \bmod K^2}{K} \right\rfloor,\quad
v = r \bmod K ,
$$

và với cột thứ $t$ trong một ảnh, $t = i\,W_{out} + j$ nên $i = \lfloor t / W_{out} \rfloor$,
$j = t \bmod W_{out}$. Phần tử tương ứng của ảnh đã đệm là
$\tilde{X}_{n,\; c,\; i s + u,\; j s + v}$. Ba mảng `k`, `i`, `j` mã hóa đúng ánh xạ này.

### 4.3 col2im là phép cộng dồn, không phải phép gán

Phép ngược `col2im` không phải là nghịch đảo đúng nghĩa. Vì một điểm ảnh xuất hiện ở nhiều cột
khác nhau của $X_{col}$ (nó nằm trong nhiều cửa sổ chồng lấn), gradient chảy ngược về nó phải
được **cộng dồn** chứ không ghi đè:

$$
\frac{\partial L}{\partial \tilde{X}_{n,c,y,x}}
\;=\;
\sum_{(r,\,t)\;\mapsto\;(n,c,y,x)} \frac{\partial L}{\partial X_{col}[r,\,t]} .
$$

Đây chính là lý do cài đặt dùng `np.add.at` thay vì phép gán thông thường. Nếu dùng phép gán,
chỉ lần ghi cuối cùng còn sống sót và gradient sẽ sai ở mọi vị trí bị chồng lấn, một lỗi rất khó
phát hiện nếu không có kiểm tra sai phân hữu hạn ở mục 9.

In [3]:

def get_im2col_indices(x_shape, KH, KW, padding=1, stride=1):
    '''Dựng ba mảng chỉ số (k, i, j) mô tả vị trí kênh / hàng / cột của mọi phần tử
    trong ma trận cột. Chỉ phụ thuộc hình dạng nên có thể tính một lần và dùng lại.'''
    N, C, H, W = x_shape
    out_h = (H + 2 * padding - KH) // stride + 1
    out_w = (W + 2 * padding - KW) // stride + 1

    # offset hàng trong cửa sổ: lặp cho đủ KW cột rồi nhân bản cho C kênh
    i0 = np.tile(np.repeat(np.arange(KH), KW), C)
    # vị trí hàng của từng cửa sổ trên ảnh
    i1 = stride * np.repeat(np.arange(out_h), out_w)
    # offset cột trong cửa sổ
    j0 = np.tile(np.arange(KW), KH * C)
    j1 = stride * np.tile(np.arange(out_w), out_h)

    i = i0.reshape(-1, 1) + i1.reshape(1, -1)      # (C*KH*KW, out_h*out_w)
    j = j0.reshape(-1, 1) + j1.reshape(1, -1)
    k = np.repeat(np.arange(C), KH * KW).reshape(-1, 1)
    return k.astype(np.intp), i.astype(np.intp), j.astype(np.intp), out_h, out_w


def im2col_indices(x, KH, KW, padding=1, stride=1):
    '''Trải mọi khối cục bộ thành cột. Trả về ma trận (C*KH*KW, N*out_h*out_w).'''
    p = padding
    x_padded = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant') if p > 0 else x
    k, i, j, out_h, out_w = get_im2col_indices(x.shape, KH, KW, padding, stride)
    cols = x_padded[:, k, i, j]                    # (N, C*KH*KW, out_h*out_w)
    C = x.shape[1]
    cols = cols.transpose(1, 2, 0).reshape(C * KH * KW, -1)
    return cols, out_h, out_w


def col2im_indices(cols, x_shape, KH, KW, padding=1, stride=1):
    '''Phép ngược của im2col: CỘNG DỒN gradient về đúng vị trí điểm ảnh gốc.'''
    N, C, H, W = x_shape
    p = padding
    x_padded = np.zeros((N, C, H + 2 * p, W + 2 * p), dtype=cols.dtype)
    k, i, j, out_h, out_w = get_im2col_indices(x_shape, KH, KW, padding, stride)
    cols_reshaped = cols.reshape(C * KH * KW, -1, N).transpose(2, 0, 1)
    np.add.at(x_padded, (slice(None), k, i, j), cols_reshaped)   # cộng dồn, không ghi đè
    return x_padded if p == 0 else x_padded[:, :, p:-p, p:-p]


# Kiểm tra nhanh hình dạng
_x = np.arange(1 * 2 * 4 * 4, dtype=np.float64).reshape(1, 2, 4, 4)
_cols, _oh, _ow = im2col_indices(_x, 3, 3, padding=0, stride=1)
print('Đầu vào       :', _x.shape)
print('Ma trận cột   :', _cols.shape, ' (kỳ vọng (C*K*K, N*out_h*out_w) = (18, 4))')
print('out_h, out_w  :', _oh, _ow)
print()
print('Cột đầu tiên (khối 3x3 góc trên trái của cả hai kênh):')
print(_cols[:, 0].reshape(2, 3, 3))

Đầu vào       : (1, 2, 4, 4)
Ma trận cột   : (18, 4)  (kỳ vọng (C*K*K, N*out_h*out_w) = (18, 4))
out_h, out_w  : 2 2

Cột đầu tiên (khối 3x3 góc trên trái của cả hai kênh):
[[[ 0.  1.  2.]
  [ 4.  5.  6.]
  [ 8.  9. 10.]]

 [[16. 17. 18.]
  [20. 21. 22.]
  [24. 25. 26.]]]



**Diễn giải.** Với đầu vào $(1, 2, 4, 4)$, bộ lọc $3\times3$, đệm 0 và bước 1, công thức
kích thước cho $H_{out} = W_{out} = 4 - 3 + 1 = 2$, tức 4 cửa sổ. Mỗi cửa sổ gom
$C_{in} K^2 = 2 \cdot 9 = 18$ giá trị, nên ma trận cột có dạng $(18, 4)$ đúng như in ra.
Cột đầu tiên khi được định dạng lại về $(2,3,3)$ tái hiện chính xác khối góc trên trái của hai
kênh, xác nhận ba mảng chỉ số được dựng đúng.


## 5. Ví dụ tích chập làm tay và đối chiếu với cài đặt

Để chứng minh cài đặt `im2col` cho ra đúng phép tích chập ở mục 3.1, báo cáo tính tay một ví dụ
nhỏ rồi so với đầu ra của lớp `Conv2D`.

Cho ảnh đơn kênh $4 \times 4$ và bộ lọc dò biên dọc $3 \times 3$, đệm $p = 0$, bước $s = 1$,
chệch $b = 0$:

$$
X=\begin{bmatrix}1&2&3&0\\0&1&2&3\\3&0&1&2\\2&3&0&1\end{bmatrix},
\qquad
W=\begin{bmatrix}1&0&-1\\1&0&-1\\1&0&-1\end{bmatrix}.
$$

Kích thước đầu ra: $H_{out} = W_{out} = (4 + 0 - 3)/1 + 1 = 2$.

**Ô $Z_{0,0}$** lấy khối hàng 0..2, cột 0..2:

$$
\begin{bmatrix}1&2&3\\0&1&2\\3&0&1\end{bmatrix}
\;\odot\;
\begin{bmatrix}1&0&-1\\1&0&-1\\1&0&-1\end{bmatrix}
\Rightarrow
(1-3) + (0-2) + (3-1) \;=\; -2-2+2 \;=\; \mathbf{-2}.
$$

**Ô $Z_{0,1}$** lấy khối hàng 0..2, cột 1..3:

$$
\begin{bmatrix}2&3&0\\1&2&3\\0&1&2\end{bmatrix}
\Rightarrow (2-0) + (1-3) + (0-2) \;=\; 2-2-2 \;=\; \mathbf{-2}.
$$

**Ô $Z_{1,0}$** lấy khối hàng 1..3, cột 0..2:

$$
\begin{bmatrix}0&1&2\\3&0&1\\2&3&0\end{bmatrix}
\Rightarrow (0-2) + (3-1) + (2-0) \;=\; -2+2+2 \;=\; \mathbf{2}.
$$

**Ô $Z_{1,1}$** lấy khối hàng 1..3, cột 1..3:

$$
\begin{bmatrix}1&2&3\\0&1&2\\3&0&1\end{bmatrix}
\Rightarrow (1-3) + (0-2) + (3-1) \;=\; -2-2+2 \;=\; \mathbf{-2}.
$$

Vậy kết quả tính tay là

$$
Z \;=\; \begin{bmatrix} -2 & -2 \\ \;\;2 & -2 \end{bmatrix}.
$$

Ô lớn nhất là $Z_{1,0} = 2$: đó là vị trí duy nhất mà cột trái của cửa sổ đậm hơn hẳn cột phải,
đúng với vai trò dò biên dọc của bộ lọc.

In [4]:

X_demo = np.array([[1, 2, 3, 0],
                   [0, 1, 2, 3],
                   [3, 0, 1, 2],
                   [2, 3, 0, 1]], dtype=np.float64)[None, None, :, :]   # (1,1,4,4)
W_demo = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=np.float64)[None, None, :, :]     # (1,1,3,3)

cols_demo, oh, ow = im2col_indices(X_demo, 3, 3, padding=0, stride=1)
W_row = W_demo.reshape(1, -1)
Z_demo = (W_row @ cols_demo).reshape(1, oh, ow, 1).transpose(3, 0, 1, 2)

Z_hand = np.array([[-2.0, -2.0],
                   [ 2.0, -2.0]])

print('Ma trận cột X_col (9 x 4):')
print(cols_demo.astype(int))
print()
print('Kết quả cài đặt im2col + GEMM:')
print(Z_demo[0, 0])
print()
print('Kết quả tính tay:')
print(Z_hand)
print()
print('Sai lệch tuyệt đối lớn nhất :', np.abs(Z_demo[0, 0] - Z_hand).max())
print('Khớp hoàn toàn              :', np.allclose(Z_demo[0, 0], Z_hand))

Ma trận cột X_col (9 x 4):
[[1 2 0 1]
 [2 3 1 2]
 [3 0 2 3]
 [0 1 3 0]
 [1 2 0 1]
 [2 3 1 2]
 [3 0 2 3]
 [0 1 3 0]
 [1 2 0 1]]

Kết quả cài đặt im2col + GEMM:
[[-2. -2.]
 [ 2. -2.]]

Kết quả tính tay:
[[-2. -2.]
 [ 2. -2.]]

Sai lệch tuyệt đối lớn nhất : 0.0
Khớp hoàn toàn              : True



**Diễn giải.** Ma trận cột in ra có đúng 9 hàng (vì $C_{in}K^2 = 1 \cdot 9$) và 4 cột (vì
$H_{out}W_{out} = 4$), mỗi cột là một khối $3\times3$ đã được trải phẳng theo thứ tự hàng trước
cột sau. Kết quả của đường đi `im2col` cộng GEMM trùng khít với bốn giá trị tính tay
$\{-2, -2, 2, -2\}$ với sai lệch tuyệt đối bằng 0. Đây là bằng chứng đầu tiên rằng cài đặt
hiện thực đúng định nghĩa toán học ở mục 3.1, trước khi ta kiểm chứng phần lan truyền ngược ở
mục 9.


## 6. Lan truyền ngược qua tầng tích chập

Vì phép thuận đã quy về $Z_{col} = W_{row}X_{col} + b\mathbf{1}^\top$, đạo hàm được lấy theo
quy tắc chuỗi của phép nhân ma trận. Đặt $G \equiv \partial L / \partial Z_{col}
\in \mathbb{R}^{C_{out} \times (N H_{out} W_{out})}$, ta có ba công thức:

$$
\frac{\partial L}{\partial W_{row}} \;=\; G \, X_{col}^{\top}
\;\in\; \mathbb{R}^{C_{out} \times C_{in}K^2},
\qquad
\frac{\partial L}{\partial b} \;=\; G\,\mathbf{1} \;=\; \sum_{t} G_{:,t},
$$

$$
\frac{\partial L}{\partial X_{col}} \;=\; W_{row}^{\top} G
\;\in\; \mathbb{R}^{C_{in}K^2 \times (N H_{out} W_{out})} .
$$

Nếu chọn quy ước xếp hàng chuyển vị, tức coi mỗi **hàng** là một cửa sổ,
$Z_{col}^{\top} = X_{col}^{\top} W_{row}^{\top}$, thì công thức gradient trọng số viết thành
dạng quen thuộc trong nhiều tài liệu:

$$
\frac{\partial L}{\partial W_{row}^{\top}} \;=\; X_{col}^{\top}\,\frac{\partial L}{\partial Z_{col}^{\top}} .
$$

Hai cách viết chỉ khác nhau ở bố cục bộ nhớ, giá trị số hoàn toàn trùng nhau. Cài đặt trong
notebook dùng dạng thứ nhất.

Bước cuối cùng là đưa $\partial L/\partial X_{col}$ trở lại hình dạng ảnh bằng `col2im`, tức
cộng dồn theo đúng công thức ở mục 4.3, rồi cắt bỏ viền đệm nếu $p > 0$.

Cần chú ý thứ tự chuyển vị khi định dạng $G$: khối gradient đến có dạng
$(N, C_{out}, H_{out}, W_{out})$ trong khi $Z_{col}$ xếp theo
$(C_{out}, H_{out}, W_{out}, N)$, nên phải `transpose(1, 2, 3, 0)` trước khi làm phẳng. Đây là
nguồn lỗi phổ biến nhất khi tự cài tích chập, và cũng là lỗi mà kiểm tra sai phân hữu hạn phát
hiện ngay lập tức.

In [5]:

class Conv2D:
    '''Tầng tích chập hai chiều, vector hóa hoàn toàn bằng im2col / col2im.'''

    def __init__(self, C_in, C_out, K=3, padding=1, stride=1, init='scaled', seed=0,
                 dtype=np.float32):
        rng = np.random.default_rng(seed)
        if init == 'he':
            # He Normal: sigma = sqrt(2 / fan_in), fan_in = C_in * K * K
            sigma = np.sqrt(2.0 / (C_in * K * K))
        else:
            sigma = 0.01                      # khởi tạo randn * 0.01 của Baseline
        self.W = (rng.standard_normal((C_out, C_in, K, K)) * sigma).astype(dtype)
        self.b = np.zeros(C_out, dtype=dtype)
        self.K, self.padding, self.stride = K, padding, stride
        self.C_in, self.C_out = C_in, C_out
        self.init_sigma = float(sigma)

    def params(self):
        return [('W', self.W), ('b', self.b)]

    def grads(self):
        return [('W', self.dW), ('b', self.db)]

    def forward(self, x):
        self.x_shape = x.shape
        cols, oh, ow = im2col_indices(x, self.K, self.K, self.padding, self.stride)
        self.cols = cols
        W_row = self.W.reshape(self.C_out, -1)             # (C_out, C_in*K*K)
        out = W_row @ cols + self.b.reshape(-1, 1)         # GEMM duy nhất
        # (C_out, oh, ow, N) -> (N, C_out, oh, ow)
        out = out.reshape(self.C_out, oh, ow, x.shape[0]).transpose(3, 0, 1, 2)
        return out

    def backward(self, dout):
        # dout: (N, C_out, oh, ow)
        self.db = dout.sum(axis=(0, 2, 3))
        G = dout.transpose(1, 2, 3, 0).reshape(self.C_out, -1)   # (C_out, N*oh*ow)
        self.dW = (G @ self.cols.T).reshape(self.W.shape)        # dL/dW_row = G X_col^T
        W_row = self.W.reshape(self.C_out, -1)
        dcols = W_row.T @ G                                      # dL/dX_col = W_row^T G
        return col2im_indices(dcols, self.x_shape, self.K, self.K,
                              self.padding, self.stride)

    def out_size(self, H):
        return (H + 2 * self.padding - self.K) // self.stride + 1

print('Lớp Conv2D đã được định nghĩa.')
_c = Conv2D(1, 4, 3, padding=1, init='he', seed=1, dtype=np.float64)
print('Kích thước W :', _c.W.shape, '| sigma khởi tạo He =', round(_c.init_sigma, 6),
      '| kỳ vọng sqrt(2/9) =', round(float(np.sqrt(2/9)), 6))
print('28 ->', _c.out_size(28), '(same padding giữ nguyên kích thước)')

Lớp Conv2D đã được định nghĩa.
Kích thước W : (4, 1, 3, 3) | sigma khởi tạo He = 0.471405 | kỳ vọng sqrt(2/9) = 0.471405
28 -> 28 (same padding giữ nguyên kích thước)



## 7. Các tầng còn lại: ReLU, MaxPool, Flatten, Dense

### 7.1 ReLU

$$
\mathrm{ReLU}(x) = \max(0, x),
\qquad
\frac{\partial \mathrm{ReLU}}{\partial x} =
\begin{cases} 1 & x > 0 \\ 0 & x \le 0 \end{cases}
$$

Lan truyền ngược chỉ là phép nhân theo phần tử với mặt nạ nhị phân đã ghi nhớ ở lượt thuận.
Tại $x = 0$ hàm không khả vi; ta quy ước đạo hàm bằng 0. Quy ước này ảnh hưởng tới kiểm tra sai
phân hữu hạn ở mục 9 nếu một điểm rơi đúng vào gốc, nên ở đó báo cáo chọn các chỉ số có
gradient khác 0 rõ rệt.

### 7.2 Max pooling và định tuyến gradient theo argmax

Với cửa sổ $k \times k$ bước $s$:

$$
Y_{n,c,i,j} \;=\; \max_{0 \le u,v < k} X_{n,c,\,is+u,\,js+v} .
$$

Gọi $(u^\*, v^\*)$ là vị trí đạt cực đại. Đạo hàm riêng theo một điểm đầu vào bằng 1 nếu điểm
đó chính là điểm thắng, bằng 0 nếu không:

$$
\frac{\partial Y_{n,c,i,j}}{\partial X_{n,c,\,is+u,\,js+v}}
= \mathbb{1}\!\left[(u,v) = (u^\*, v^\*)\right] .
$$

Do đó gradient đi ngược được **định tuyến** nguyên vẹn về đúng ô thắng cuộc, mọi ô khác trong
cửa sổ nhận 0:

$$
\frac{\partial L}{\partial X_{n,c,y,x}}
= \sum_{(i,j)} \frac{\partial L}{\partial Y_{n,c,i,j}}\;
\mathbb{1}\!\left[(y,x) \text{ là argmax của cửa sổ } (i,j)\right].
$$

Khi các cửa sổ không chồng lấn ($s = k$, đúng trường hợp của báo cáo) tổng trên có nhiều nhất
một số hạng khác 0. Cài đặt tận dụng lại `im2col` với $C = 1$ trên khối đã gộp trục $N$ và $C$,
lấy `argmax` theo trục cửa sổ, rồi gieo gradient vào đúng chỉ số đó trước khi gọi `col2im`.
Cách này giữ nguyên tinh thần vector hóa 100%.

### 7.3 Flatten

Chỉ định dạng lại $(N, C, H, W) \to (N, CHW)$ ở lượt thuận và đảo ngược ở lượt ngược. Không có
tham số học được, không làm thay đổi giá trị.

### 7.4 Dense

Với $Y = XW + \mathbf{1}b^\top$, $X \in \mathbb{R}^{N \times d_{in}}$,
$W \in \mathbb{R}^{d_{in} \times d_{out}}$:

$$
\frac{\partial L}{\partial W} = X^{\top}\frac{\partial L}{\partial Y},
\qquad
\frac{\partial L}{\partial b} = \mathbf{1}^{\top}\frac{\partial L}{\partial Y},
\qquad
\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y}\,W^{\top}.
$$

Khởi tạo He Normal cho tầng Dense dùng $\sigma = \sqrt{2/d_{in}}$, bản chất vẫn là
$\sqrt{2/\text{fan\_in}}$ như ở tầng tích chập.

In [6]:

class ReLU:
    def params(self): return []
    def grads(self):  return []
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, d):
        return d * self.mask


class MaxPool2D:
    '''Max pooling 2x2 bước 2, gradient định tuyến theo argmax, vector hóa qua im2col.'''
    def __init__(self, size=2, stride=2):
        self.size, self.stride = size, stride
    def params(self): return []
    def grads(self):  return []

    def forward(self, x):
        N, C, H, W = x.shape
        k, s = self.size, self.stride
        oh, ow = (H - k) // s + 1, (W - k) // s + 1
        self.x_shape = x.shape
        x_r = x.reshape(N * C, 1, H, W)               # gộp N và C để pool độc lập từng kênh
        cols, _, _ = im2col_indices(x_r, k, k, padding=0, stride=s)   # (k*k, N*C*oh*ow)
        self.cols_shape = cols.shape
        self.arg = np.argmax(cols, axis=0)            # ghi nhớ vị trí thắng cuộc
        out = cols[self.arg, np.arange(cols.shape[1])]
        return out.reshape(oh, ow, N * C).transpose(2, 0, 1).reshape(N, C, oh, ow)

    def backward(self, dout):
        N, C, H, W = self.x_shape
        k, s = self.size, self.stride
        oh, ow = dout.shape[2], dout.shape[3]
        dcols = np.zeros(self.cols_shape, dtype=dout.dtype)
        d_flat = dout.reshape(N * C, oh, ow).transpose(1, 2, 0).ravel()
        dcols[self.arg, np.arange(dcols.shape[1])] = d_flat   # chỉ ô argmax nhận gradient
        dx = col2im_indices(dcols, (N * C, 1, H, W), k, k, padding=0, stride=s)
        return dx.reshape(N, C, H, W)


class Flatten:
    def params(self): return []
    def grads(self):  return []
    def forward(self, x):
        self.shape = x.shape
        return x.reshape(x.shape[0], -1)
    def backward(self, d):
        return d.reshape(self.shape)


class Dense:
    def __init__(self, n_in, n_out, init='scaled', seed=0, dtype=np.float32):
        rng = np.random.default_rng(seed)
        sigma = np.sqrt(2.0 / n_in) if init == 'he' else 0.01
        self.W = (rng.standard_normal((n_in, n_out)) * sigma).astype(dtype)
        self.b = np.zeros(n_out, dtype=dtype)
        self.init_sigma = float(sigma)
    def params(self): return [('W', self.W), ('b', self.b)]
    def grads(self):  return [('W', self.dW), ('b', self.db)]
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    def backward(self, d):
        self.dW = self.x.T @ d
        self.db = d.sum(axis=0)
        return d @ self.W.T


# Kiểm chứng nhanh định tuyến gradient của MaxPool trên một ví dụ 4x4 tính tay
xp = np.array([[1., 5., 2., 0.],
               [3., 2., 1., 4.],
               [0., 1., 9., 2.],
               [7., 3., 2., 2.]])[None, None]
mp = MaxPool2D(2, 2)
yp = mp.forward(xp)
print('Đầu vào 4x4:\n', xp[0, 0])
print('Sau max pool 2x2 (kỳ vọng [[5,4],[7,9]]):\n', yp[0, 0])
gp = mp.backward(np.ones_like(yp))
print('Gradient định tuyến ngược (1 tại đúng 4 ô thắng cuộc, 0 ở các ô còn lại):\n', gp[0, 0])
print('Tổng gradient =', gp.sum(), '(phải bằng số ô đầu ra = 4)')

Đầu vào 4x4:
 [[1. 5. 2. 0.]
 [3. 2. 1. 4.]
 [0. 1. 9. 2.]
 [7. 3. 2. 2.]]
Sau max pool 2x2 (kỳ vọng [[5,4],[7,9]]):
 [[5. 4.]
 [7. 9.]]
Gradient định tuyến ngược (1 tại đúng 4 ô thắng cuộc, 0 ở các ô còn lại):
 [[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]
Tổng gradient = 4.0 (phải bằng số ô đầu ra = 4)



**Diễn giải.** Bốn cửa sổ $2\times2$ không chồng lấn của ma trận ví dụ có cực đại lần lượt là
5, 4, 7 và 9, đúng bằng đầu ra mà `MaxPool2D` trả về. Khi lan truyền ngược với gradient đến toàn
bằng 1, ma trận gradient thu được chỉ có bốn số 1 nằm đúng tại bốn ô chứa cực đại, mọi ô khác
bằng 0, và tổng gradient bằng 4 đúng bằng số phần tử đầu ra. Điều này xác nhận cơ chế định
tuyến theo argmax đã trình bày ở mục 7.2 được cài đặt chính xác.


## 8. Softmax, entropy chéo và thuật toán Adam

### 8.1 Softmax với thủ thuật trừ cực đại

Với logit $z \in \mathbb{R}^{C}$, softmax cho phân phối xác suất

$$
\hat{y}_k \;=\; \frac{e^{z_k}}{\sum_{j=1}^{C} e^{z_j}} .
$$

Cài đặt trực tiếp công thức này rất dễ tràn số: nếu $z_k$ đạt cỡ 800 thì $e^{z_k}$ vượt ngưỡng
biểu diễn của số thực 64 bit và cho `inf`, kéo theo `nan` ở bước chia. Thủ thuật chuẩn là trừ
đi cực đại $m = \max_j z_j$ trước khi lấy lũy thừa. Phép trừ này **không làm đổi giá trị** vì

$$
\frac{e^{z_k - m}}{\sum_j e^{z_j - m}}
= \frac{e^{z_k} e^{-m}}{e^{-m}\sum_j e^{z_j}}
= \frac{e^{z_k}}{\sum_j e^{z_j}} ,
$$

nhưng bảo đảm mũ lớn nhất luôn bằng $e^{0} = 1$, nên không bao giờ tràn trên, còn tràn dưới thì
vô hại vì chỉ cho ra 0 ở các thành phần vốn đã không đáng kể.

### 8.2 Entropy chéo và gradient hợp nhất

Với nhãn one-hot $y$, hàm mất mát của một mẫu là
$L = -\sum_{k} y_k \log \hat{y}_k = -\log \hat{y}_{k^\*}$ với $k^\*$ là lớp đúng.

Đạo hàm của softmax theo logit là
$\partial \hat{y}_k / \partial z_j = \hat{y}_k(\delta_{kj} - \hat{y}_j)$, một ma trận Jacobi
$C \times C$. Nếu lập trình hai tầng tách rời, ta phải nhân với Jacobi này ở mỗi bước, vừa tốn
kém vừa dễ mất ổn định số. Khi **hợp nhất** softmax với entropy chéo, chuỗi rút gọn ngoạn mục:

$$
\frac{\partial L}{\partial z_j}
= \sum_k \frac{\partial L}{\partial \hat{y}_k}\frac{\partial \hat{y}_k}{\partial z_j}
= \sum_k \left(-\frac{y_k}{\hat{y}_k}\right)\hat{y}_k(\delta_{kj} - \hat{y}_j)
= -y_j + \hat{y}_j \sum_k y_k
$$

và vì $\sum_k y_k = 1$ ta được kết quả gọn gàng nổi tiếng

$$
\boxed{\;\frac{\partial L}{\partial z} \;=\; \hat{y} - y\;}
$$

Với một lô $N$ mẫu và mất mát lấy trung bình, gradient là $(\hat{Y} - Y)/N$. Đây chính là dạng
được cài trong hàm `softmax_cross_entropy` bên dưới: không cần dựng Jacobi, không có phép chia
cho $\hat{y}_k$ nên không có nguy cơ chia cho số gần 0.

### 8.3 Adam

Adam kết hợp mô-men bậc nhất và bậc hai có hiệu chỉnh chệch:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^{2},
$$

$$
\hat{m}_t = \frac{m_t}{1-\beta_1^{t}}, \qquad
\hat{v}_t = \frac{v_t}{1-\beta_2^{t}}, \qquad
\theta_t = \theta_{t-1} - \eta\,\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}.
$$

Hệ số hiệu chỉnh $1-\beta_1^t$ và $1-\beta_2^t$ là cần thiết vì $m_0 = v_0 = 0$ khiến các bước
đầu bị lệch mạnh về 0. Báo cáo dùng $\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$,
$\epsilon = 10^{-8}$, đúng giá trị mặc định của bài báo gốc và của cả PyTorch lẫn Keras, nhờ đó
so sánh ba cách cài đặt ở notebook 02 là công bằng về mặt tối ưu.

In [7]:

def softmax(z):
    '''Softmax ổn định số: trừ cực đại theo từng hàng trước khi lấy lũy thừa.'''
    z_shift = z - z.max(axis=1, keepdims=True)
    e = np.exp(z_shift)
    return e / e.sum(axis=1, keepdims=True)


def softmax_cross_entropy(z, y):
    '''Trả về (loss trung bình, dL/dz, xác suất dự đoán).
    Gradient hợp nhất: dL/dz = (y_hat - y_onehot) / N.'''
    p = softmax(z)
    N = z.shape[0]
    loss = float(-np.log(np.clip(p[np.arange(N), y], 1e-12, None)).mean())
    dz = p.copy()
    dz[np.arange(N), y] -= 1.0
    dz /= N
    return loss, dz, p


class Adam:
    def __init__(self, layers, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.layers = layers
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [{n: np.zeros_like(p) for n, p in l.params()} for l in layers]
        self.v = [{n: np.zeros_like(p) for n, p in l.params()} for l in layers]
        self.t = 0

    def step(self, lr=None):
        lr = self.lr if lr is None else lr
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t
        bc2 = 1.0 - self.b2 ** self.t
        for li, layer in enumerate(self.layers):
            gd = dict(layer.grads())
            for name, p in layer.params():
                g = gd[name]
                self.m[li][name] = self.b1 * self.m[li][name] + (1 - self.b1) * g
                self.v[li][name] = self.b2 * self.v[li][name] + (1 - self.b2) * (g * g)
                m_hat = self.m[li][name] / bc1
                v_hat = self.v[li][name] / bc2
                p -= lr * m_hat / (np.sqrt(v_hat) + self.eps)   # cập nhật tại chỗ


# Kiểm chứng tính ổn định của thủ thuật trừ cực đại
z_big = np.array([[1000.0, 999.0, 998.0]])
with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(z_big) / np.exp(z_big).sum(axis=1, keepdims=True)
print('Softmax cài ngây thơ trên logit lớn :', naive)
print('Softmax có trừ cực đại              :', softmax(z_big))
print('Tổng xác suất                       :', softmax(z_big).sum())

Softmax cài ngây thơ trên logit lớn :

 [[nan nan nan]]
Softmax có trừ cực đại              : [[0.66524096 0.24472847 0.09003057]]
Tổng xác suất                       : 0.9999999999999999



**Diễn giải.** Với logit $[1000, 999, 998]$, cách cài ngây thơ cho `nan` vì $e^{1000}$ vượt
ngưỡng biểu diễn của kiểu `float64` (giới hạn xấp xỉ $1{,}8 \times 10^{308}$, tương ứng
$e^{709}$). Cách cài có trừ cực đại trả về phân phối hợp lệ với tổng đúng bằng 1. Trong huấn
luyện thực tế logit hiếm khi lớn tới mức đó, nhưng khi mạng bắt đầu phân kỳ, giá trị logit tăng
rất nhanh và chính lúc đó thủ thuật này ngăn toàn bộ quá trình sụp thành `nan`.


## 9. Ghép mạng và kiểm tra gradient bằng sai phân hữu hạn

### 9.1 Hai kiến trúc theo hợp đồng

| | Baseline | Improved |
|---|---|---|
| Đệm | $p = 0$ (valid) | $p = 1$ (same) |
| Khởi tạo | $\mathcal{N}(0,1) \times 0{,}01$ | He Normal $\sigma=\sqrt{2/\mathrm{fan\_in}}$ |
| Learning rate | cố định | có lịch giảm theo bậc thang |
| Luồng kích thước | $28 \to 26 \to 13 \to 11 \to 5$ | $28 \to 28 \to 14 \to 14 \to 7$ |
| Số chiều sau Flatten | $16 \times 5 \times 5 = 400$ | $16 \times 7 \times 7 = 784$ |

Cả hai đều theo đúng chuỗi tầng mà mục 4 hợp đồng quy định:
`Conv2D(8,K=3)` → ReLU → MaxPool(2) → `Conv2D(16,K=3)` → ReLU → MaxPool(2) → Flatten →
`Dense(64)` → ReLU → `Dense(10)` → Softmax.

### 9.2 Nguyên lý kiểm tra gradient

Đây là phần kiểm chứng quan trọng nhất của notebook. Với mỗi tham số vô hướng $\theta$, sai
phân hữu hạn trung tâm cho xấp xỉ

$$
\frac{\partial L}{\partial \theta} \;\approx\;
\frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon}
\;+\; O(\varepsilon^{2}) .
$$

Sai số bậc $O(\varepsilon^2)$ của công thức trung tâm tốt hơn hẳn công thức tiến
$\left(L(\theta+\varepsilon) - L(\theta)\right)/\varepsilon$ vốn chỉ đạt $O(\varepsilon)$. Ta so
gradient số này với gradient giải tích thu được từ lan truyền ngược bằng **sai số tương đối**

$$
\mathrm{relerr} \;=\;
\frac{\left| g_{\text{giải tích}} - g_{\text{số}} \right|}
     {\max\!\left(10^{-12},\; \left| g_{\text{giải tích}} \right| + \left| g_{\text{số}} \right|\right)} .
$$

Quy ước đánh giá thông dụng: `relerr` $< 10^{-7}$ là xuất sắc, $< 10^{-5}$ là chấp nhận được,
$> 10^{-3}$ gần như chắc chắn có lỗi cài đặt.

Ba lưu ý thực hành. Thứ nhất, toàn bộ phép kiểm tra chạy ở `float64`; với `float32` nhiễu làm
tròn cỡ $10^{-7}$ sẽ nhấn chìm tín hiệu. Thứ hai, ta chọn $\varepsilon = 10^{-5}$ để cân bằng
giữa sai số cắt cụt (giảm khi $\varepsilon$ nhỏ) và sai số làm tròn (tăng khi $\varepsilon$
nhỏ). Thứ ba, ReLU không khả vi tại 0 nên ta ưu tiên kiểm tra tại các chỉ số có gradient giải
tích lớn, tránh vùng gãy.

In [8]:

class CNN:
    '''Mạng tích chập hai chiều: Conv-ReLU-Pool x2 -> Flatten -> Dense-ReLU -> Dense.'''

    def __init__(self, padding=0, init='scaled', seed=RANDOM_SEED,
                 c1=8, c2=16, n_hidden=64, dtype=np.float32, img=28):
        self.conv1 = Conv2D(1,  c1, 3, padding, init=init, seed=seed,     dtype=dtype)
        self.conv2 = Conv2D(c1, c2, 3, padding, init=init, seed=seed + 1, dtype=dtype)
        s1 = self.conv1.out_size(img);  s2 = s1 // 2
        s3 = self.conv2.out_size(s2);   s4 = s3 // 2
        self.flat_dim = c2 * s4 * s4
        self.shape_trace = [img, s1, s2, s3, s4, self.flat_dim]
        self.fc1 = Dense(self.flat_dim, n_hidden, init=init, seed=seed + 2, dtype=dtype)
        self.fc2 = Dense(n_hidden, 10,          init=init, seed=seed + 3, dtype=dtype)
        self.relu1, self.relu2, self.relu3 = ReLU(), ReLU(), ReLU()
        self.pool1, self.pool2 = MaxPool2D(2, 2), MaxPool2D(2, 2)
        self.flatten = Flatten()
        self.seq = [self.conv1, self.relu1, self.pool1,
                    self.conv2, self.relu2, self.pool2,
                    self.flatten, self.fc1, self.relu3, self.fc2]
        self.learnables = [('conv1', self.conv1), ('conv2', self.conv2),
                           ('fc1', self.fc1),     ('fc2', self.fc2)]

    def forward(self, x):
        for layer in self.seq:
            x = layer.forward(x)
        return x

    def backward(self, dz):
        for layer in reversed(self.seq):
            dz = layer.backward(dz)
        return dz

    def n_params(self):
        return int(sum(p.size for _, l in self.learnables for _, p in l.params()))


net_base = CNN(padding=0, init='scaled')
net_impr = CNN(padding=1, init='he')
for tag, net in [('Baseline', net_base), ('Improved', net_impr)]:
    t = net.shape_trace
    print(f'{tag:9s} luồng kích thước: {t[0]} -> conv1 {t[1]} -> pool {t[2]} '
          f'-> conv2 {t[3]} -> pool {t[4]} -> flatten {t[5]}  | tham số = {net.n_params():,}')

Baseline  luồng kích thước: 28 -> conv1 26 -> pool 13 -> conv2 11 -> pool 5 -> flatten 400  | tham số = 27,562
Improved  luồng kích thước: 28 -> conv1 28 -> pool 14 -> conv2 14 -> pool 7 -> flatten 784  | tham số = 52,138


In [9]:

def gradient_check(net, X, y, eps=1e-5, n_per_tensor=2, seed=RANDOM_SEED):
    '''So sánh gradient giải tích với gradient sai phân hữu hạn trung tâm
    trên MỌI tensor tham số học được của mạng.'''
    rng = np.random.default_rng(seed)

    # 1. Một lượt thuận + ngược để lấy gradient giải tích
    z = net.forward(X)
    loss, dz, _ = softmax_cross_entropy(z, y)
    net.backward(dz)

    rows = []
    for lname, layer in net.learnables:
        gd = dict(layer.grads())
        for pname, P in layer.params():
            G = gd[pname]
            flat_g = G.ravel()
            # chọn chỉ số có |gradient| lớn nhất + một chỉ số ngẫu nhiên trong top 20%
            order = np.argsort(-np.abs(flat_g))
            cand = [order[0]]
            top = order[:max(2, len(order) // 5)]
            while len(cand) < n_per_tensor:
                pick = int(rng.choice(top))
                if pick not in cand:
                    cand.append(pick)
            for flat_idx in cand:
                idx = np.unravel_index(flat_idx, P.shape)
                old = P[idx]
                P[idx] = old + eps
                lp, _, _ = softmax_cross_entropy(net.forward(X), y)
                P[idx] = old - eps
                lm, _, _ = softmax_cross_entropy(net.forward(X), y)
                P[idx] = old                      # phục hồi nguyên trạng
                g_num = (lp - lm) / (2 * eps)
                g_ana = float(G[idx])
                rel = abs(g_ana - g_num) / max(1e-12, abs(g_ana) + abs(g_num))
                rows.append((lname, pname, tuple(int(v) for v in idx),
                             g_ana, float(g_num), rel))
    return loss, rows


# Kiểm tra trên mạng Improved ở độ chính xác float64 với 8 ảnh MNIST thật
rng_gc = np.random.default_rng(RANDOM_SEED)
gc_idx = rng_gc.choice(len(X_tr), size=8, replace=False)
X_gc = X_tr[gc_idx].astype(np.float64)
y_gc = y_tr[gc_idx]

net_gc = CNN(padding=1, init='he', dtype=np.float64, seed=RANDOM_SEED)
loss_gc, rows = gradient_check(net_gc, X_gc, y_gc, eps=1e-5, n_per_tensor=2)

print(f'Mất mát tại điểm kiểm tra: {loss_gc:.10f}   (log(10) = {math.log(10):.6f} '
      f'là mức kỳ vọng khi mạng chưa học)')
print()
hdr = f"{'Tầng':<7} {'Tham số':<8} {'Chỉ số':<16} {'Gradient giải tích':>22} {'Gradient số':>22} {'Sai số tương đối':>18}"
print(hdr); print('-' * len(hdr))
worst = 0.0
for lname, pname, idx, ga, gn, rel in rows:
    worst = max(worst, rel)
    print(f'{lname:<7} {pname:<8} {str(idx):<16} {ga:>22.15f} {gn:>22.15f} {rel:>18.3e}')
print('-' * len(hdr))
print(f'Sai số tương đối lớn nhất trên toàn bộ {len(rows)} phép kiểm tra: {worst:.3e}')
print('Kết luận:', 'ĐẠT (mọi sai số < 1e-7)' if worst < 1e-7 else
      ('CHẤP NHẬN ĐƯỢC (< 1e-5)' if worst < 1e-5 else 'KHÔNG ĐẠT, cài đặt có lỗi'))

Mất mát tại điểm kiểm tra: 4.8711983482   (log(10) = 2.302585 là mức kỳ vọng khi mạng chưa học)



Tầng    Tham số  Chỉ số               Gradient giải tích            Gradient số   Sai số tương đối
--------------------------------------------------------------------------------------------------
conv1   W        (3, 0, 1, 0)          1.338484631992061      1.338484632018577          9.905e-12
conv1   W        (3, 0, 0, 0)          1.300926250637802      1.300926250547008          3.490e-11
conv1   b        (3,)                  0.653509558916325      0.653509558956600          3.081e-11
conv1   b        (5,)                  0.459814161659594      0.459814161635208          2.652e-11
conv2   W        (6, 3, 1, 0)          0.829958094179735      0.829958094161753          1.083e-11
conv2   W        (9, 1, 0, 0)          0.265357856061037      0.265357856088499          5.175e-11
conv2   b        (6,)                  0.435617464220576      0.435617464233928          1.533e-11
conv2   b        (12,)                -0.360198154781942     -0.360198154814029          4.454e-11
fc1     


**Diễn giải bảng kiểm tra gradient.** Bảng trên đối chiếu gradient giải tích do phép lan truyền
ngược viết tay sinh ra với gradient số tính bằng sai phân hữu hạn trung tâm, trên tất cả tám
tensor tham số học được của mạng: `conv1.W`, `conv1.b`, `conv2.W`, `conv2.b`, `fc1.W`, `fc1.b`,
`fc2.W`, `fc2.b`. Mỗi tensor được kiểm hai vị trí nên tổng cộng có 16 phép so sánh.

Sai số tương đối lớn nhất trên cả 16 phép kiểm tra chỉ là $1{,}861 \times 10^{-10}$, đạt được ở
ô `fc1.W[139, 34]`. Giá trị nhỏ nhất là $8{,}241 \times 10^{-12}$ tại `fc2.b[1]`. Toàn bộ dải sai
số nằm trong khoảng $[8{,}2\times10^{-12};\ 1{,}9\times10^{-10}]$, tức thấp hơn ngưỡng xuất sắc
$10^{-7}$ từ hai tới bốn bậc độ lớn, và đúng bằng thang nhiễu làm tròn của số thực 64 bit chứ
không phải sai lệch công thức. Đáng chú ý là hai ô có sai số lớn nhất đều là những ô có gradient
giải tích nhỏ nhất trong bảng ($-0{,}1029$ và $0{,}0898$), phù hợp với lý thuyết: sai số tương đối
của sai phân hữu hạn tăng khi tín hiệu gradient giảm, vì phần mẫu số của công thức relerr nhỏ đi
trong khi nhiễu làm tròn giữ nguyên.

Kết luận: **toàn bộ chuỗi đạo hàm viết tay là đúng**, bao gồm ba thành phần khó nhất là phép cộng
dồn trong `col2im`, phép định tuyến argmax của max pooling, và các phép chuyển vị khi định dạng
lại gradient của tầng tích chập. Bất kỳ sai lệch nào ở ba chỗ đó, kể cả một phép `transpose` đặt
nhầm thứ tự, đều sẽ đẩy sai số tương đối lên thang $10^{-1}$ và bị bảng này bắt được ngay.

Một chi tiết cần giải thích trung thực: mất mát tại điểm kiểm tra là $4{,}8712$, **cao hơn** mức
$\log 10 \approx 2{,}3026$ ứng với phân phối đều trên mười lớp. Điều này không phải dấu hiệu sai
sót. Khởi tạo He Normal cố ý giữ phương sai tín hiệu qua từng tầng nên logit ở đầu ra đã có biên
độ đáng kể ngay từ đầu; mạng vì vậy đưa ra dự đoán **tự tin nhưng ngẫu nhiên**, và entropy chéo
phạt nặng những dự đoán tự tin mà sai, đẩy mất mát vượt trên mốc phân phối đều. Ngược lại, nhật
ký huấn luyện của biến thể Baseline ở mục 11 khởi đầu với mất mát trung bình epoch đầu chỉ
$1{,}0703$, vì khởi tạo $\times 0{,}01$ làm logit gần như triệt tiêu và softmax cho ra phân phối
gần đều. Hai hành vi trái ngược này chính là minh chứng định lượng cho tác động của lựa chọn
khởi tạo mà mục 12 sẽ khai thác.


## 10. Vòng lặp huấn luyện

Vòng lặp dùng gradient descent theo lô nhỏ với Adam. Mỗi epoch dữ liệu được xáo trộn lại để
tránh mạng học thuộc thứ tự mẫu. Sau mỗi epoch, báo cáo đo mất mát và độ chính xác trên cả tập
train con lẫn tập validation, và ghi nhớ bộ trọng số tại epoch có độ chính xác validation cao
nhất. Việc chọn epoch tốt nhất **chỉ dựa vào validation**, tập kiểm thử tuyệt đối không tham gia.

Với biến thể Improved, learning rate giảm theo bậc thang $\eta_e = \eta_0 \cdot \gamma^{
\lfloor e / T \rfloor}$ với $\gamma = 0{,}5$ và $T = 4$. Lý do: giai đoạn đầu cần bước lớn để
thoát nhanh khỏi vùng khởi tạo, giai đoạn sau cần bước nhỏ để tinh chỉnh quanh cực tiểu mà
không dao động qua lại.

In [10]:

def evaluate(net, X, y, batch=256, return_prob=False):
    '''Đánh giá theo lô để không làm tràn bộ nhớ khi chạy trên 10 000 ảnh.'''
    losses, probs = [], []
    for s in range(0, len(X), batch):
        xb, yb = X[s:s + batch], y[s:s + batch]
        z = net.forward(xb)
        l, _, p = softmax_cross_entropy(z, yb)
        losses.append(l * len(xb))
        probs.append(p)
    P = np.concatenate(probs, axis=0)
    loss = float(np.sum(losses) / len(X))
    pred = P.argmax(axis=1)
    acc = float((pred == y).mean())
    return (loss, acc, pred, P) if return_prob else (loss, acc)


def snapshot(net):
    return {name: {pn: p.copy() for pn, p in l.params()} for name, l in net.learnables}


def restore(net, snap):
    for name, l in net.learnables:
        for pn, p in l.params():
            p[...] = snap[name][pn]


def train_model(net, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
                lr0=1e-3, lr_decay=None, decay_every=4, tag='model', seed=RANDOM_SEED):
    '''Huấn luyện một mạng CNN. lr_decay=None nghĩa là learning rate cố định.'''
    rng = np.random.default_rng(seed)
    opt = Adam([l for _, l in net.learnables], lr=lr0)
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_acc, best_epoch, best_snap = -1.0, 0, None
    t_start = time.time()
    n = len(X_tr)

    for ep in range(1, epochs + 1):
        lr_ep = lr0 if lr_decay is None else lr0 * (lr_decay ** ((ep - 1) // decay_every))
        perm = rng.permutation(n)
        run_loss, run_correct, seen = 0.0, 0, 0
        for s in range(0, n, batch):
            bidx = perm[s:s + batch]
            xb, yb = X_tr[bidx], y_tr[bidx]
            z = net.forward(xb)
            loss, dz, p = softmax_cross_entropy(z, yb)
            net.backward(dz)
            opt.step(lr_ep)
            run_loss += loss * len(bidx)
            run_correct += int((p.argmax(axis=1) == yb).sum())
            seen += len(bidx)
        tr_loss, tr_acc = run_loss / seen, run_correct / seen
        va_loss, va_acc = evaluate(net, X_va, y_va)

        hist['train_loss'].append(float(tr_loss)); hist['val_loss'].append(float(va_loss))
        hist['train_acc'].append(float(tr_acc));   hist['val_acc'].append(float(va_acc))
        hist['lr'].append(float(lr_ep))
        star = ''
        if va_acc > best_acc:
            best_acc, best_epoch, best_snap = va_acc, ep, snapshot(net)
            star = '  <-- tốt nhất'
        print(f'[{tag}] epoch {ep:2d}/{epochs} | lr={lr_ep:.5f} | '
              f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
              f'val_loss={va_loss:.4f} val_acc={va_acc:.4f}{star}')

    train_time = time.time() - t_start
    restore(net, best_snap)
    print(f'[{tag}] hoàn tất sau {train_time:.1f}s | epoch tốt nhất = {best_epoch} '
          f'| val_acc tốt nhất = {best_acc:.4f}')
    return hist, best_epoch, train_time


## 11. Huấn luyện biến thể Baseline

Cấu hình: đệm 0, khởi tạo $\mathcal{N}(0,1)\times 0{,}01$, learning rate cố định $10^{-3}$.

In [11]:

np.random.seed(RANDOM_SEED)
net_base = CNN(padding=0, init='scaled', seed=RANDOM_SEED, dtype=np.float32)
print(f'Baseline: {net_base.n_params():,} tham số | '
      f'sigma khởi tạo conv1 = {net_base.conv1.init_sigma}')
print(f'Huấn luyện trên tập con {len(X_tr)} ảnh, validation {len(X_va)} ảnh')
print('-' * 100)
hist_base, best_ep_base, time_base = train_model(
    net_base, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
    lr0=1e-3, lr_decay=None, tag='Baseline')

Baseline: 27,562 tham số | sigma khởi tạo conv1 = 0.01
Huấn luyện trên tập con 12000 ảnh, validation 3000 ảnh
----------------------------------------------------------------------------------------------------


[Baseline] epoch  1/12 | lr=0.00100 | train_loss=1.0703 train_acc=0.6374 | val_loss=0.4334 val_acc=0.8690  <-- tốt nhất


[Baseline] epoch  2/12 | lr=0.00100 | train_loss=0.3531 train_acc=0.8928 | val_loss=0.3308 val_acc=0.8947  <-- tốt nhất


[Baseline] epoch  3/12 | lr=0.00100 | train_loss=0.2590 train_acc=0.9210 | val_loss=0.2412 val_acc=0.9263  <-- tốt nhất


[Baseline] epoch  4/12 | lr=0.00100 | train_loss=0.2065 train_acc=0.9350 | val_loss=0.1949 val_acc=0.9393  <-- tốt nhất


...

[Baseline] epoch  8/12 | lr=0.00100 | train_loss=0.1067 train_acc=0.9653 | val_loss=0.1417 val_acc=0.9560


[Baseline] epoch  9/12 | lr=0.00100 | train_loss=0.0977 train_acc=0.9693 | val_loss=0.1235 val_acc=0.9650  <-- tốt nhất


[Baseline] epoch 10/12 | lr=0.00100 | train_loss=0.0879 train_acc=0.9694 | val_loss=0.1133 val_acc=0.9630


[Baseline] epoch 11/12 | lr=0.00100 | train_loss=0.0859 train_acc=0.9715 | val_loss=0.1082 val_acc=0.9660  <-- tốt nhất


[Baseline] epoch 12/12 | lr=0.00100 | train_loss=0.0755 train_acc=0.9739 | val_loss=0.1098 val_acc=0.9657
[Baseline] hoàn tất sau 120.0s | epoch tốt nhất = 11 | val_acc tốt nhất = 0.9660



## 12. Huấn luyện biến thể Improved

Ba thay đổi so với Baseline, mỗi thay đổi nhắm vào một điểm yếu cụ thể.

**Đệm same ($p=1$).** Với đệm 0, mỗi tầng tích chập bào mòn hai điểm ảnh ở mỗi chiều, và điểm
ảnh ở sát biên chỉ được một số ít cửa sổ nhìn thấy nên đóng góp ít vào biểu diễn. Đệm same giữ
nguyên kích thước, bảo toàn thông tin biên và nâng số chiều sau Flatten từ 400 lên 784, cho
tầng phân loại nhiều tín hiệu hơn.

**Khởi tạo He Normal.** Khởi tạo $\times 0{,}01$ khiến phương sai tín hiệu suy giảm theo cấp số
nhân qua từng tầng: với fan-in bằng 9 ở `conv1` và 72 ở `conv2`, hệ số khuếch đại phương sai
xấp xỉ $\text{fan\_in} \times 0{,}01^2$, tức nhỏ hơn 1 rất nhiều. He Normal chọn
$\sigma = \sqrt{2/\text{fan\_in}}$ để phương sai được bảo toàn qua mỗi tầng ReLU, trong đó hệ số
2 bù cho việc ReLU triệt tiêu một nửa số kích hoạt.

**Lịch giảm learning rate.** $\eta_0 = 2\times10^{-3}$ giảm một nửa sau mỗi 4 epoch.

In [12]:

np.random.seed(RANDOM_SEED)
net_impr = CNN(padding=1, init='he', seed=RANDOM_SEED, dtype=np.float32)
print(f'Improved: {net_impr.n_params():,} tham số')
print(f'  sigma He conv1 = {net_impr.conv1.init_sigma:.6f}  (sqrt(2/9)   = {np.sqrt(2/9):.6f})')
print(f'  sigma He conv2 = {net_impr.conv2.init_sigma:.6f}  (sqrt(2/72)  = {np.sqrt(2/72):.6f})')
print(f'  sigma He fc1   = {net_impr.fc1.init_sigma:.6f}  (sqrt(2/784) = {np.sqrt(2/784):.6f})')
print('-' * 100)
hist_impr, best_ep_impr, time_impr = train_model(
    net_impr, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
    lr0=2e-3, lr_decay=0.5, decay_every=4, tag='Improved')

Improved: 52,138 tham số
  sigma He conv1 = 0.471405  (sqrt(2/9)   = 0.471405)
  sigma He conv2 = 0.166667  (sqrt(2/72)  = 0.166667)
  sigma He fc1   = 0.050508  (sqrt(2/784) = 0.050508)
----------------------------------------------------------------------------------------------------


[Improved] epoch  1/12 | lr=0.00200 | train_loss=0.4181 train_acc=0.8698 | val_loss=0.1611 val_acc=0.9467  <-- tốt nhất


[Improved] epoch  2/12 | lr=0.00200 | train_loss=0.1111 train_acc=0.9648 | val_loss=0.0942 val_acc=0.9710  <-- tốt nhất


[Improved] epoch  3/12 | lr=0.00200 | train_loss=0.0692 train_acc=0.9767 | val_loss=0.0891 val_acc=0.9703


[Improved] epoch  4/12 | lr=0.00200 | train_loss=0.0528 train_acc=0.9822 | val_loss=0.0790 val_acc=0.9757  <-- tốt nhất


...

[Improved] epoch  8/12 | lr=0.00100 | train_loss=0.0079 train_acc=0.9987 | val_loss=0.0718 val_acc=0.9797  <-- tốt nhất


[Improved] epoch  9/12 | lr=0.00050 | train_loss=0.0047 train_acc=0.9997 | val_loss=0.0715 val_acc=0.9803  <-- tốt nhất


[Improved] epoch 10/12 | lr=0.00050 | train_loss=0.0032 train_acc=0.9999 | val_loss=0.0716 val_acc=0.9807  <-- tốt nhất


[Improved] epoch 11/12 | lr=0.00050 | train_loss=0.0029 train_acc=0.9998 | val_loss=0.0753 val_acc=0.9797


[Improved] epoch 12/12 | lr=0.00050 | train_loss=0.0024 train_acc=1.0000 | val_loss=0.0725 val_acc=0.9800
[Improved] hoàn tất sau 148.0s | epoch tốt nhất = 10 | val_acc tốt nhất = 0.9807



## 13. Đánh giá trên trọn vẹn 10 000 ảnh của tập kiểm thử

In [13]:

def full_metrics(net, X, y, name):
    t0 = time.time()
    loss, acc, pred, P = evaluate(net, X, y, batch=512, return_prob=True)
    prec, rec, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    cm = confusion_matrix(y, pred, labels=list(range(10)))
    per_class = (cm.diagonal() / cm.sum(axis=1)).astype(float)
    conf = P.max(axis=1)
    wrong = np.where(pred != y)[0]
    order = wrong[np.argsort(-conf[wrong])][:8]
    hce = [{'index': int(i), 'true': int(y[i]), 'pred': int(pred[i]),
            'confidence': float(conf[i])} for i in order]
    print(f'--- {name} trên {len(y)} ảnh kiểm thử (suy luận {time.time()-t0:.1f}s) ---')
    print(f'  loss            = {loss:.6f}')
    print(f'  accuracy        = {acc:.6f}  ({acc*100:.2f}%)')
    print(f'  macro precision = {prec:.6f}')
    print(f'  macro recall    = {rec:.6f}')
    print(f'  macro F1        = {f1:.6f}')
    print(f'  số ảnh sai      = {len(wrong)}')
    return dict(loss=float(loss), accuracy=float(acc), macro_precision=float(prec),
                macro_recall=float(rec), macro_f1=float(f1),
                confusion_matrix=cm.tolist(), per_class_accuracy=per_class.tolist(),
                high_conf_errors=hce, pred=pred)

res_base = full_metrics(net_base, X_te, y_te, 'Baseline (NumPy)')
print()
res_impr = full_metrics(net_impr, X_te, y_te, 'Improved (NumPy)')

gain = (res_impr['accuracy'] - res_base['accuracy']) * 100
gain_f1 = (res_impr['macro_f1'] - res_base['macro_f1']) * 100
err_drop = (1 - res_impr['accuracy']) / max(1e-12, (1 - res_base['accuracy']))
print()
print('=' * 70)
print(f'Cải thiện tuyệt đối về accuracy : {gain:+.2f} điểm phần trăm '
      f'({res_base["accuracy"]*100:.2f}% -> {res_impr["accuracy"]*100:.2f}%)')
print(f'Cải thiện tuyệt đối về macro F1 : {gain_f1:+.2f} điểm phần trăm')
print(f'Tỉ lệ lỗi còn lại so với Baseline: {err_drop:.3f} '
      f'(giảm {100*(1-err_drop):.1f}% số lỗi)')
print('=' * 70)

--- Baseline (NumPy) trên 10000 ảnh kiểm thử (suy luận 1.8s) ---
  loss            = 0.092433
  accuracy        = 0.970300  (97.03%)
  macro precision = 0.970198
  macro recall    = 0.970106
  macro F1        = 0.970091
  số ảnh sai      = 297



--- Improved (NumPy) trên 10000 ảnh kiểm thử (suy luận 3.6s) ---
  loss            = 0.062125
  accuracy        = 0.982300  (98.23%)
  macro precision = 0.982298
  macro recall    = 0.982112
  macro F1        = 0.982176
  số ảnh sai      = 177

Cải thiện tuyệt đối về accuracy : +1.20 điểm phần trăm (97.03% -> 98.23%)
Cải thiện tuyệt đối về macro F1 : +1.21 điểm phần trăm
Tỉ lệ lỗi còn lại so với Baseline: 0.596 (giảm 40.4% số lỗi)


In [14]:

print('Báo cáo phân loại chi tiết của biến thể Improved trên 10 000 ảnh kiểm thử:')
print(classification_report(y_te, res_impr['pred'], digits=4, zero_division=0))

Báo cáo phân loại chi tiết của biến thể Improved trên 10 000 ảnh kiểm thử:
              precision    recall  f1-score   support

           0     0.9770    0.9949    0.9858       980
           1     0.9878    0.9956    0.9917      1135
           2     0.9882    0.9767    0.9825      1032
           3     0.9736    0.9861    0.9798      1010
           4     0.9867    0.9857    0.9862       982
           5     0.9853    0.9765    0.9809       892
           6     0.9874    0.9802    0.9838       958
           7     0.9777    0.9796    0.9786      1028
           8     0.9765    0.9805    0.9785       974
           9     0.9828    0.9653    0.9740      1009

    accuracy                         0.9823     10000
   macro avg     0.9823    0.9821    0.9822     10000
weighted avg     0.9823    0.9823    0.9823     10000




## 14. Hình 1: đường cong huấn luyện Baseline so với Improved

In [15]:

ep_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_axis, hist_base['train_loss'], 'o-', color='#2E86C1', label='Baseline train')
axes[0].plot(ep_axis, hist_base['val_loss'],  'o--', color='#2E86C1', alpha=0.55, label='Baseline val')
axes[0].plot(ep_axis, hist_impr['train_loss'], 's-', color='#C0392B', label='Improved train')
axes[0].plot(ep_axis, hist_impr['val_loss'],  's--', color='#C0392B', alpha=0.55, label='Improved val')
axes[0].set_title('Mất mát entropy chéo theo epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mất mát'); axes[0].set_yscale('log')
axes[0].legend(fontsize=9); axes[0].set_xticks(ep_axis)

axes[1].plot(ep_axis, np.array(hist_base['train_acc'])*100, 'o-', color='#2E86C1', label='Baseline train')
axes[1].plot(ep_axis, np.array(hist_base['val_acc'])*100,  'o--', color='#2E86C1', alpha=0.55, label='Baseline val')
axes[1].plot(ep_axis, np.array(hist_impr['train_acc'])*100, 's-', color='#C0392B', label='Improved train')
axes[1].plot(ep_axis, np.array(hist_impr['val_acc'])*100,  's--', color='#C0392B', alpha=0.55, label='Improved val')
axes[1].axvline(best_ep_base, color='#2E86C1', ls=':', lw=1.2)
axes[1].axvline(best_ep_impr, color='#C0392B', ls=':', lw=1.2)
axes[1].set_title('Độ chính xác theo epoch (đường chấm dọc = epoch tốt nhất)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Độ chính xác (%)')
axes[1].legend(fontsize=9, loc='lower right'); axes[1].set_xticks(ep_axis)

fig.suptitle(f'CNN NumPy thuần trên MNIST: Baseline so với Improved\n'
             f'(huấn luyện trên tập con phân tầng {len(X_tr)} ảnh, validation {len(X_va)} ảnh)',
             fontsize=14, y=1.04)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_scratch_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_scratch_curves.png')
print(f'Mất mát validation cuối cùng: Baseline {hist_base["val_loss"][-1]:.4f} | '
      f'Improved {hist_impr["val_loss"][-1]:.4f}')
print(f'Độ chính xác validation tốt nhất: Baseline {max(hist_base["val_acc"])*100:.2f}% | '
      f'Improved {max(hist_impr["val_acc"])*100:.2f}%')

Đã lưu fig_mnist_scratch_curves.png
Mất mát validation cuối cùng: Baseline 0.1098 | Improved 0.0725
Độ chính xác validation tốt nhất: Baseline 96.60% | Improved 98.07%



**Diễn giải hình `fig_mnist_scratch_curves.png`.** Hai bảng nhật ký huấn luyện ở mục 11 và mục 12
cùng với hình này cho thấy ba hiện tượng rõ rệt.

**Tốc độ hội tụ chênh lệch rất lớn ở giai đoạn đầu.** Sau đúng một epoch, biến thể Improved đã
đạt độ chính xác validation $94{,}67\%$, trong khi Baseline mới chỉ đạt $86{,}90\%$, chênh nhau
$7{,}77$ điểm phần trăm. Mức $94{,}67\%$ mà Improved đạt trong một epoch, Baseline phải đi tới
epoch 4 mới vượt qua ($93{,}93\%$ ở epoch 4 và $94{,}77\%$ ở epoch 5). Nguyên nhân trực tiếp là
khởi tạo: với $\sigma = 0{,}01$, tín hiệu lan truyền xuôi bị nén lại qua từng tầng, gradient ban
đầu cực nhỏ, và Adam phải mất vài trăm bước lặp mới tích lũy đủ mô-men bậc hai để đưa bước cập
nhật lên biên độ hữu ích. He Normal với $\sigma = \sqrt{2/9} = 0{,}4714$ ở `conv1` bảo toàn phương
sai ngay từ bước đầu tiên.

**Lịch giảm learning rate tạo ra hai bậc gãy rõ rệt trên đường mất mát.** Trên trục log của panel
trái, mất mát huấn luyện của Improved rơi đột ngột đúng tại epoch 5 (từ $0{,}0528$ xuống
$0{,}0232$, giảm 56%) và một lần nữa tại epoch 9 (từ $0{,}0079$ xuống $0{,}0047$), trùng khít với
hai thời điểm $\eta$ bị chia đôi theo công thức $\eta_e = 2\times10^{-3}\cdot 0{,}5^{\lfloor
(e-1)/4\rfloor}$. Đây là biểu hiện kinh điển của việc thu nhỏ bước cập nhật giúp mô hình lắng
xuống đáy một lòng chảo mà trước đó nó chỉ dao động quanh miệng.

**Hai chế độ khớp dữ liệu trái ngược nhau.** Baseline kết thúc epoch 12 với độ chính xác train
$97{,}39\%$ và validation $96{,}57\%$, khoảng cách chỉ $0{,}82$ điểm phần trăm, và cả hai đường vẫn
đang đi lên. Đó là trạng thái **chưa khớp đủ**: mô hình còn dư năng lực học nhưng bị chặn bởi tốc
độ hội tụ. Improved thì ngược lại, đạt $100{,}00\%$ trên tập train ở epoch 12 trong khi validation
dừng ở $98{,}00\%$, khoảng cách $2{,}00$ điểm phần trăm, tức đã bắt đầu **khớp quá**. Tuy nhiên
mức khớp quá này còn lành tính vì mất mát validation không tăng trở lại mà đi ngang trong dải
$[0{,}0715;\ 0{,}0753]$ suốt bốn epoch cuối. Cơ chế chọn epoch theo độ chính xác validation cao
nhất đã giữ lại trọng số tại epoch 10 ($98{,}07\%$) thay vì trọng số cuối cùng, nên phần khớp quá
không làm hỏng kết quả kiểm thử.

Cũng cần lưu ý rằng mọi con số trên hình này đến từ tập con phân tầng 12 000 ảnh train và 3 000
ảnh validation, không phải toàn bộ 48 000 ảnh, đúng như ghi chú ngân sách tính toán ở đầu notebook.


## 15. Hình 2: ma trận nhầm lẫn của hai biến thể

In [16]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, res, name in [(axes[0], res_base, 'Baseline'), (axes[1], res_impr, 'Improved')]:
    cm = np.array(res['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                annot_kws={'size': 8}, linewidths=0.4, linecolor='#DDDDDD')
    ax.set_title(f'{name}: accuracy = {res["accuracy"]*100:.2f}%, '
                 f'{int(cm.sum() - np.trace(cm))} ảnh sai', fontsize=12)
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
fig.suptitle('Ma trận nhầm lẫn trên 10 000 ảnh kiểm thử, CNN NumPy thuần', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_scratch_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_scratch_confusion.png')

# Liệt kê các cặp nhầm lẫn nặng nhất của biến thể Improved
cm_i = np.array(res_impr['confusion_matrix'])
off = [(cm_i[a, b], a, b) for a in range(10) for b in range(10) if a != b and cm_i[a, b] > 0]
off.sort(reverse=True)
print()
print('Sáu cặp nhầm lẫn nặng nhất của Improved (nhãn thật -> nhãn dự đoán):')
for n, a, b in off[:6]:
    print(f'  {a} -> {b} : {n} ảnh  ({100*n/cm_i[a].sum():.2f}% số ảnh lớp {a})')

Đã lưu fig_mnist_scratch_confusion.png

Sáu cặp nhầm lẫn nặng nhất của Improved (nhãn thật -> nhãn dự đoán):
  9 -> 7 : 10 ảnh  (0.99% số ảnh lớp 9)
  6 -> 0 : 9 ảnh  (0.94% số ảnh lớp 6)
  5 -> 3 : 9 ảnh  (1.01% số ảnh lớp 5)
  5 -> 6 : 8 ảnh  (0.90% số ảnh lớp 5)
  4 -> 9 : 8 ảnh  (0.81% số ảnh lớp 4)
  9 -> 4 : 7 ảnh  (0.69% số ảnh lớp 9)



**Diễn giải hình `fig_mnist_scratch_confusion.png`.** Trên cùng 10 000 ảnh kiểm thử, Baseline sai
297 ảnh còn Improved chỉ sai 177 ảnh, tức loại bỏ được 120 lỗi, tương đương giảm $40{,}4\%$ tổng
số lỗi. Đây là cách đọc ý nghĩa hơn so với mức tăng $1{,}20$ điểm phần trăm về độ chính xác: khi
đã ở vùng trên 97%, mỗi điểm phần trăm tương ứng với một phần rất lớn của khối lỗi còn lại.

Sáu cặp nhầm lẫn nặng nhất của Improved là $9 \to 7$ với 10 ảnh, $6 \to 0$ với 9 ảnh, $5 \to 3$
với 9 ảnh, $5 \to 6$ với 8 ảnh, $4 \to 9$ với 8 ảnh và $9 \to 4$ với 7 ảnh. Ba trong sáu cặp này
đã được dự đoán từ trước ở notebook 00 dựa trên quan sát thị giác lưới mẫu: cặp 4 với 9 xuất hiện
đối xứng ở cả hai chiều ($4 \to 9$ và $9 \to 4$, tổng 15 ảnh) đúng vì hai chữ số chỉ khác nhau ở
chỗ vòng đầu có khép kín hay không, còn cặp 5 với 3 chia sẻ phần thân cong bên phải. Cặp
$6 \to 0$ thì không nằm trong dự đoán ban đầu; giải thích hợp lý là khi nét mở đầu của số 6 được
viết ngắn, phần còn lại chỉ là một vòng kín gần như không phân biệt được với số 0.

Đọc theo lớp, bảng phân loại chi tiết cho thấy chữ số 1 dễ nhất với recall $99{,}56\%$, phù hợp
với nhận xét ở notebook 00 rằng đây là lớp có lượng mực thấp nhất và hình dạng đơn giản nhất.
Khó nhất là chữ số 9 với recall chỉ $96{,}53\%$, chính là lớp phân tán lỗi sang cả 7 và 4. Khoảng
cách giữa lớp dễ nhất và lớp khó nhất là $3{,}03$ điểm phần trăm, một biên độ hẹp khẳng định mô
hình không bỏ rơi lớp nào.

So sánh hai ma trận cạnh nhau còn cho thấy cải thiện phân bố khá đều trên đường chéo chứ không
tập trung vào một lớp duy nhất, nghĩa là ba thay đổi của biến thể Improved tác động lên chất
lượng biểu diễn tổng thể chứ không phải chỉ sửa được một kiểu lỗi cục bộ.


## 16. Hình 3: so sánh bốn chỉ số giữa hai biến thể

In [17]:

labels = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
vals_b = [res_base['accuracy'], res_base['macro_precision'],
          res_base['macro_recall'], res_base['macro_f1']]
vals_i = [res_impr['accuracy'], res_impr['macro_precision'],
          res_impr['macro_recall'], res_impr['macro_f1']]

xpos = np.arange(len(labels)); w = 0.36
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(xpos - w/2, np.array(vals_b)*100, w, label='Baseline (p=0, randn*0.01, lr cố định)',
            color='#2E86C1', edgecolor='black', linewidth=0.6)
b2 = ax.bar(xpos + w/2, np.array(vals_i)*100, w, label='Improved (p=1, He Normal, lr giảm dần)',
            color='#C0392B', edgecolor='black', linewidth=0.6)
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.12,
                f'{bar.get_height():.2f}', ha='center', fontsize=9)
for k in range(len(labels)):
    d = (vals_i[k] - vals_b[k]) * 100
    ax.annotate(f'{d:+.2f} đpt', xy=(xpos[k], max(vals_b[k], vals_i[k])*100 + 1.0),
                ha='center', fontsize=10, color='#196F3D', fontweight='bold')
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel('Giá trị (%)')
ax.set_ylim(min(min(vals_b), min(vals_i))*100 - 3, 103)
ax.set_title('CNN NumPy thuần trên MNIST: Baseline so với Improved\n'
             f'(đánh giá trên 10 000 ảnh kiểm thử; huấn luyện trên tập con {len(X_tr)} ảnh)',
             fontsize=13)
ax.legend(fontsize=9, loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_scratch_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_scratch_comparison.png')
print()
print(f"{'Chỉ số':<18}{'Baseline':>12}{'Improved':>12}{'Chênh lệch (đpt)':>20}")
print('-' * 62)
for k, lb in enumerate(labels):
    print(f'{lb:<18}{vals_b[k]*100:>11.2f}%{vals_i[k]*100:>11.2f}%{(vals_i[k]-vals_b[k])*100:>19.2f}')

Đã lưu fig_mnist_scratch_comparison.png

Chỉ số                Baseline    Improved    Chênh lệch (đpt)
--------------------------------------------------------------
Accuracy                97.03%      98.23%               1.20
Macro Precision         97.02%      98.23%               1.21
Macro Recall            97.01%      98.21%               1.20
Macro F1                97.01%      98.22%               1.21



**Diễn giải hình `fig_mnist_scratch_comparison.png`.** Bốn chỉ số tăng gần như bằng nhau:
accuracy $+1{,}20$ đpt, macro precision $+1{,}21$ đpt, macro recall $+1{,}20$ đpt và macro F1
$+1{,}21$ đpt. Sự đồng đều này mang một thông điệp phương pháp luận quan trọng. Nếu cải thiện chỉ
đến từ việc mô hình ưu ái các lớp đông mẫu, độ chính xác tổng thể sẽ tăng nhiều hơn hẳn macro F1,
vì macro F1 lấy trung bình không trọng số trên mười lớp. Việc bốn cột tăng song song cho thấy
biến thể Improved nâng chất lượng trên toàn bộ mười lớp chứ không đánh đổi lớp này lấy lớp khác.

Ngoài ra, với Improved thì accuracy $98{,}23\%$ và macro F1 $98{,}22\%$ chênh nhau chỉ $0{,}01$
điểm phần trăm. Đây là hệ quả trực tiếp của kết luận ở notebook 00 rằng MNIST gần cân bằng lớp với
tỉ số mất cân bằng chỉ 1,2724 trên tập kiểm thử: khi các lớp có kích thước tương đương, trung bình
có trọng số và trung bình không trọng số hội tụ về cùng một giá trị.

Cái giá của cải thiện là chi phí tính toán và số tham số. Improved có 52 138 tham số so với 27 562
của Baseline, tức gấp 1,89 lần, và toàn bộ phần chênh nằm ở tầng `fc1` do đệm same nâng số chiều
sau Flatten từ 400 lên 784. Thời gian huấn luyện cũng tăng từ 120,0 giây lên 148,0 giây, khoảng
23%, vì đệm same khiến bản đồ đặc trưng lớn hơn ở mọi tầng. Đổi $23\%$ thời gian và $89\%$ tham số
để lấy $40{,}4\%$ số lỗi là một đánh đổi rõ ràng có lợi trong bối cảnh bài toán này.


## 17. Lưu kết quả trung gian cho notebook 02

Hợp đồng quy định miền MNIST chỉ có **một** tệp `reports/metrics_mnist.json` duy nhất, được
lắp ráp ở notebook 02 sau khi đã có đủ kết quả của cả bốn mô hình. Notebook 01 vì vậy ghi kết
quả của hai mô hình NumPy ra một tệp trung gian `reports/metrics_mnist_scratch_partial.json`
để notebook 02 nạp lại và trộn vào tệp cuối cùng. Tệp trung gian này không thay thế và không
mâu thuẫn với tệp bắt buộc.

In [18]:

def pack(res, hist, best_ep, ttime, net, epochs, framework):
    return {
        'framework': framework,
        'params': int(net.n_params()),
        'train_time_s': float(ttime),
        'epochs': int(epochs),
        'best_epoch': int(best_ep),
        'accuracy': res['accuracy'],
        'macro_precision': res['macro_precision'],
        'macro_recall': res['macro_recall'],
        'macro_f1': res['macro_f1'],
        'loss': res['loss'],
        'history': {'train_loss': hist['train_loss'], 'val_loss': hist['val_loss'],
                    'train_acc': hist['train_acc'],  'val_acc': hist['val_acc']},
        'confusion_matrix': res['confusion_matrix'],
        'per_class_accuracy': res['per_class_accuracy'],
        'high_conf_errors': res['high_conf_errors'],
    }

partial = {
    'numpy_baseline': pack(res_base, hist_base, best_ep_base, time_base, net_base,
                           EPOCHS, 'NumPy From Scratch (Baseline)'),
    'numpy_improved': pack(res_impr, hist_impr, best_ep_impr, time_impr, net_impr,
                           EPOCHS, 'NumPy From Scratch (Improved)'),
    'subset': {'n_train_subset': int(len(X_tr)), 'n_val_subset': int(len(X_va)),
               'n_test': int(len(X_te)), 'batch_size': BATCH_SIZE, 'epochs': EPOCHS},
    'preprocess': {'MEAN': MEAN, 'STD': STD},
    'gradient_check': {
        'eps': 1e-5, 'dtype': 'float64', 'n_checks': len(rows),
        'max_rel_error': float(max(r[5] for r in rows)),
        'rows': [{'layer': r[0], 'param': r[1], 'index': list(r[2]),
                  'analytic': r[3], 'numeric': r[4], 'rel_error': r[5]} for r in rows],
    },
}
out_path = os.path.join(REP_DIR, 'metrics_mnist_scratch_partial.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(partial, f, ensure_ascii=False, indent=2)
print('Đã ghi', out_path, f'({os.path.getsize(out_path)/1024:.1f} KB)')
print()
for f in ['fig_mnist_scratch_curves.png', 'fig_mnist_scratch_confusion.png',
          'fig_mnist_scratch_comparison.png']:
    p = os.path.join(FIG_DIR, f)
    print(f'  {f:40s} {os.path.getsize(p)/1024:7.1f} KB  tồn tại={os.path.exists(p)}')

Đã ghi ../reports\metrics_mnist_scratch_partial.json (13.4 KB)

  fig_mnist_scratch_curves.png               161.2 KB  tồn tại=True
  fig_mnist_scratch_confusion.png             87.1 KB  tồn tại=True
  fig_mnist_scratch_comparison.png            64.4 KB  tồn tại=True



## 18. Kết luận notebook 01

Notebook này đã hoàn thành ba nhiệm vụ đặt ra ở phần mục tiêu.

**Về tính đúng đắn.** Phép tích chập được kiểm chứng hai lớp. Lớp thứ nhất là ví dụ số làm tay
trên ma trận $4\times4$ với bộ lọc dò biên dọc, cho kết quả
$Z = \begin{bmatrix}-2 & -2\\ 2 & -2\end{bmatrix}$ trùng khít tuyệt đối với đầu ra của đường đi
`im2col` cộng GEMM. Lớp thứ hai, quan trọng hơn, là kiểm tra sai phân hữu hạn trung tâm trên tất
cả tám tensor tham số học được với sai số tương đối lớn nhất chỉ $1{,}861 \times 10^{-10}$. Kết
quả này chứng minh phép lan truyền ngược viết tay, bao gồm cộng dồn `col2im`, định tuyến argmax
của max pooling và gradient hợp nhất softmax cộng entropy chéo $\partial L/\partial z = \hat{y}-y$,
là đúng về mặt toán học chứ không phải chỉ "chạy được".

**Về hiệu năng.** Biến thể Baseline đạt $97{,}03\%$ độ chính xác và $0{,}9701$ macro F1 trên trọn
vẹn 10 000 ảnh kiểm thử. Biến thể Improved đạt $98{,}23\%$ và $0{,}9822$, tức cải thiện $+1{,}20$
điểm phần trăm về độ chính xác và $+1{,}21$ điểm phần trăm về macro F1, tương đương xóa bỏ
$40{,}4\%$ số lỗi (từ 297 ảnh sai xuống 177 ảnh sai). Cần nhấn mạnh rằng con số này đạt được khi
chỉ huấn luyện trên **một phần tư** dữ liệu train sẵn có (12 000 trên 48 000 ảnh), nên nó là cận
dưới của năng lực kiến trúc chứ không phải giới hạn của nó.

**Về ba yếu tố cải thiện.** Đệm same bảo toàn thông tin biên và nâng số chiều biểu diễn từ 400 lên
784. Khởi tạo He Normal bảo toàn phương sai tín hiệu, rút ngắn thời gian hội tụ từ bốn epoch xuống
một epoch để cùng đạt mốc $94{,}67\%$ validation. Lịch giảm learning rate tạo hai bậc gãy giảm mất
mát tại epoch 5 và epoch 9. Ba yếu tố này không loại trừ nhau mà bổ trợ nhau.

**Hạn chế đã ghi nhận.** Thứ nhất, do ngân sách CPU, hai mô hình NumPy chỉ dùng tập con phân tầng
12 000 ảnh train và 3 000 ảnh validation; sai lệch này được ghi vào khóa `notes` của tệp metrics
cuối cùng. Thứ hai, biến thể Improved đã chạm $100{,}00\%$ trên tập train nên phần dung lượng mô
hình còn lại không thể khai thác thêm nếu không bổ sung chính quy hóa. Chính hai hạn chế đó là
động lực cho notebook 02, nơi PyTorch và Keras sẽ dùng toàn bộ 48 000 ảnh cùng Batch Normalization
và Dropout để đẩy độ chính xác lên ngưỡng 99%.